In [2]:
from google.colab import drive,userdata
import os
import sys

drive.mount("/content/drive")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
sys.path.insert(0,"/content/drive/MyDrive/code-debugger")

Mounted at /content/drive


In [3]:
import torch
device="cuda" if torch.cuda.is_available() else "cpu"
print(f"GPU available: {torch.cuda.is_available()}")
print(f"device:{device}")

GPU available: True
device:cuda


**IMPORTS**

In [4]:
import json
import os
import time
import sys

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import get_peft_model,LoraConfig,TaskType

**Load Dataset**

In [5]:
dataset_path="/content/drive/MyDrive/code-debugger/ml_service/model/full_dataset.json"
with open(dataset_path,"r") as f:
  data=json.load(f)
LABELS = ["Syntax Error", "Runtime Error", "Logic Bug", "Multiple Issues"]
print(f"Loaded {len(data)} examples")

Loaded 232 examples


**Tokenizer**

In [6]:
BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
  return tokenizer(batch["code"],truncation=True,max_length=256,padding=False)
dataset=Dataset.from_list(data)
tokenized = dataset.map(tokenize, batched=True, remove_columns=["code"])
split = tokenized.train_test_split(test_size=0.2, seed=42)
print(f"✅ Train: {len(split['train'])}  |  Eval: {len(split['test'])}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

✅ Train: 185  |  Eval: 47


**Lora Config**

In [7]:
base_model=AutoModelForSequenceClassification.from_pretrained(BASE_MODEL,num_labels=len(LABELS))
lora_config=LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
)
model=get_peft_model(base_model,lora_config)
model.print_trainable_parameters()







model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948


**Training**

In [11]:
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = float((preds == labels).mean())
    return {"accuracy": acc}

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/code-debugger/ml_service/model/lora_weights",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    logging_steps=5,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training complete")

Epoch,Training Loss,Validation Loss,Accuracy
1,1.300773,1.200994,0.617021
2,0.588745,0.324303,0.914894
3,0.520142,0.262254,0.914894
4,0.337751,0.200711,0.936170
5,0.124026,0.197834,0.936170
6,0.006275,0.211855,0.957447
7,0.063447,0.237412,0.936170
8,0.013807,0.232958,0.936170
9,0.078322,0.262486,0.936170
10,0.003337,0.254152,0.936170


✅ Training complete


**Save the model**

In [13]:
save_path="/content/drive/MyDrive/code-debugger/ml_service/model/lora_weights"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path} ")
files=os.listdir(save_path)
for file in files:
  print(file)


Model saved to /content/drive/MyDrive/code-debugger/ml_service/model/lora_weights 
checkpoint-47
checkpoint-94
checkpoint-141
checkpoint-188
checkpoint-235
checkpoint-282
checkpoint-329
checkpoint-376
checkpoint-423
checkpoint-470
README.md
adapter_model.safetensors
adapter_config.json
tokenizer_config.json
tokenizer.json


**TEST**

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig
import torch

LABELS = ["Syntax Error", "Runtime Error", "Logic Bug", "Multiple Issues"]
save_path = "/content/drive/MyDrive/code-debugger/ml_service/model/lora_weights"

# Reload from saved weights
config = PeftConfig.from_pretrained(save_path)
base = AutoModelForSequenceClassification.from_pretrained(
    config.base_model_name_or_path, num_labels=len(LABELS)
)
loaded_model = PeftModel.from_pretrained(base, save_path)
loaded_model.eval()
loaded_tokenizer = AutoTokenizer.from_pretrained(save_path)

# Test on 4 examples — one per bug type
tests = [
    ("def greet(name)\n    print('hi')", "Syntax Error"),
    ("x = [1,2,3]\nprint(x[99])", "Runtime Error"),
    ("def is_even(n):\n    return n % 2 == 1", "Logic Bug"),
    ("def foo(n)\n    if n = 0:\n        return 1", "Multiple Issues"),
]

print("Test results:\n")
for code, expected in tests:
    inputs = loaded_tokenizer(code, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = loaded_model(**inputs).logits
    pred = LABELS[torch.argmax(logits).item()]
    status = "good" if pred == expected else "catastrophic"
    print(f"{status} Expected: {expected:<20} Got: {pred}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test results:

good Expected: Syntax Error         Got: Syntax Error
good Expected: Runtime Error        Got: Runtime Error
good Expected: Logic Bug            Got: Logic Bug
catastrophic Expected: Multiple Issues      Got: Logic Bug
